v1 : bbox roi with dicomsdl

v2-v3: 1024*512 
* 1024*512 
* efficient
* 15 epoch completed
* bathc = 8
* acc = 4

v4-v5- v6-v7 -v8 :2048* 1024

* efficient
* bathc = 8
* acc = 4
* v7 : from epoch 23 

In [ ]:
!pip install --no-index --find-links=/kaggle/input/datasets/malihealavi/pydicom-packages-rsna-breast-cancer pydicom pylibjpeg pylibjpeg-libjpeg
!pip install dicomsdl

In [ ]:
# -----------------------------------------
#           Library Installation
# -----------------------------------------

import warnings
warnings.filterwarnings("ignore")
import os
import sys
import json
from glob import glob
import random
import collections
import time
import re
from typing import Optional
import math
import pickle
from sklearn.metrics import precision_recall_curve
import gc

import numpy as np
import pandas as pd
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

import torch
from torch import nn
from torch.utils import data as torch_data
from sklearn import model_selection as sk_model_selection
from torch.nn import functional as torch_functional
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data.distributed import DistributedSampler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import train_test_split
from IPython.display import clear_output
clear_output()


# -----------------------------------------
#              Global Values
# -----------------------------------------
IMG_SIZE = 1024 
BATCH_SIZE = 8
MAX_IMAGES = 2 # Number of views per laterlaity per patient
NEG_TO_POS_RATIO = 5
N_SHARDS = 3
ACCUMULATION_STEP = 4
VIEW_TYPE_MAP = {'CC': 0, 'MLO': 1,'ML': 2 ,'LM': 3, 'AT': 4 ,'LMO': 5}  # extend / remap if other view strings (ML, LM, AT...) appear in your data
WARMUP_EPOCHS = 4
PATIENCE = 8

# -----------------------------------------
#    SubSample Train with original ratio
# -----------------------------------------

def build_epoch_shards(df, label_col='cancer', n_shards=5, seed=42):
   
    skf = StratifiedKFold(n_splits=n_shards, shuffle=True, random_state=seed)
    return {i: df.index[test_idx].tolist()
            for i, (_, test_idx) in enumerate(skf.split(df, df[label_col]))}

def get_epoch_subset(df, n_shards, epoch, base_seed=42):
    cycle, shard_idx = divmod(epoch, n_shards)
    shard_map = build_epoch_shards(df, n_shards=n_shards, seed=base_seed + cycle)
    return df.loc[shard_map[shard_idx]].reset_index(drop=True)

# -----------------------------------------
#             Balance DF
# -----------------------------------------
class BalancedPatientSampler:
    def __init__(self, neg_patient_ids, seed=42):
        self.seed = seed
        self.neg_ids = sorted(neg_patient_ids)
        self.rng = random.Random(seed)
        self.rng.shuffle(self.neg_ids)
        self.ptr = 0

    def sample(self, n_needed):
        out = []
        while n_needed > 0:
            remaining = len(self.neg_ids) - self.ptr
            take = min(remaining, n_needed)
            out.extend(self.neg_ids[self.ptr : self.ptr + take])
            self.ptr += take
            n_needed -= take
            if self.ptr >= len(self.neg_ids):
                self.rng.shuffle(self.neg_ids)
                self.ptr = 0
        return out

    def state_dict(self):
        return {
            'neg_ids': list(self.neg_ids),
            'rng_state': pickle.dumps(self.rng.getstate()), 
            'ptr': self.ptr,
        }
        
    def load_state_dict(self, state):
        try:
            self.neg_ids = list(state['neg_ids'])
            self.rng.setstate(pickle.loads(state['rng_state']))
            self.ptr = state['ptr']
        except Exception as e:
            print(f"[warn] neg_sampler state incompatible ({e}); reseeding fresh.")


def create_balanced_patient_subset(df, neg_sampler, ratio):
    patient_targets = df.groupby('patient_id')['cancer'].max()
    pos_patient_ids = sorted(patient_targets[patient_targets == 1].index.tolist())
    n_neg_needed = len(pos_patient_ids) * ratio
    sampled_neg_patient_ids = neg_sampler.sample(n_neg_needed)

    balanced_patient_ids = pos_patient_ids + sampled_neg_patient_ids
    balanced_df = df[df['patient_id'].isin(balanced_patient_ids)].copy()
    return balanced_df

def group_views_by_patient_laterality(df):
    df = df.copy()
    df['_view_rank'] = df['view'].map(VIEW_TYPE_MAP).fillna(6)
    df = df.sort_values(['patient_id', 'laterality', '_view_rank', 'image_id'])
    return df.groupby(['patient_id', 'laterality'], sort=False).agg({
        'image_id': list,
        'view': list,
        'bbox': list,
        'cancer': 'max',
    }).reset_index()
# -----------------------------------------
#             Seed Everything
# -----------------------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = False   
        torch.backends.cudnn.benchmark = True      

seed_everything()

# -----------------------------------------
#            Logit Adjustment
# -----------------------------------------
def logit_adjustment(raw_logits, p_train, p_real, pos_weight, eps=1e-8):
    """ 
       p_train      # empirical positive fraction in your actual downsampled training data
       p_real       # true prevalence
       pos_weight   # whatever value you passed to BCEWithLogitsLoss
       
    """
    train_odds = pos_weight * (p_train / (1 - p_train + eps))
    real_odds = p_real / (1 - p_real + eps)
    adjust = math.log(real_odds + eps) - math.log(train_odds + eps)
    return raw_logits + adjust   # still logits — sigmoid applied by caller
    
# -----------------------------------------
#              Augmentation
# -----------------------------------------

train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, border_mode=0, value=0, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.5),
    A.RandomGamma(gamma_limit=(95, 105), p=0.2),
    A.GaussNoise(var_limit=(1, 5), p=0.2),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.4),
    # random erase
    # A.CoarseDropout(max_holes=6, max_height=0.15, max_width=0.25, min_holes=1, min_height=0.05, min_width=0.1,
    #                 fill_value=0, mask_fill_value=None, p=0.25),
    # ], p=0.9)
    A.Resize(height= 2*IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485], std=[0.229]),
    ToTensorV2(),
] )

valid_transforms = A.Compose([
        A.Resize(height=2*IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485], std=[0.229]),
    ToTensorV2(),
] )

# -----------------------------------------
#              Preprocessing
# -----------------------------------------
def normalizer(image, photometric, dcm_meta=None):
    image = image.astype(np.float32)

    if dcm_meta is not None:
        try:
            image = apply_voi_lut(image, dcm_meta)
        except Exception:
            pass  # missing/malformed VOI LUT tags -> fall back to raw values

    if photometric == 'MONOCHROME1':
        image = image.max() - image   # invert AFTER windowing

    image_min, image_max = np.min(image), np.max(image)
    if image_max - image_min > 0:
        image = (image - image_min) / (image_max - image_min) * 255.0
    else:
        image = np.zeros_like(image)
    return image.astype(np.uint8)

    
def get_anatomy_bbox(image, threshold_value=15):
    """
    Finds the bounding box of the main anatomy in a normalized 8-bit image
    by isolating the largest connected component.
    
    Returns:
        bbox (list): [x_min, y_min, x_max, y_max]
    """
    # 1. Light Gaussian blur to smooth out internal textures and noise
    blurred = cv2.GaussianBlur(image, (5, 5), 0)
    
    # 2. Binarize the image
    # Any pixel brighter than threshold_value becomes 255 (white), rest become 0 (black)
    _, binary_mask = cv2.threshold(blurred, threshold_value, 255, cv2.THRESH_BINARY)
    
    # 3. Clean up the mask using Morphological Closing
    # This fills in small holes inside the anatomy (like dark spots in lungs or ventricles)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    closed_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
    
    # 4. Find all external contours in the mask
    contours, _ = cv2.findContours(closed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Fallback if image is completely black or no contours found
    if not contours:
        h, w = image.shape
        return [0, 0, w, h]
        
    # 5. Extract the largest contour by area (this drops small text/markers)
    largest_contour = max(contours, key=cv2.contourArea)
    
    # 6. Get the standard bounding box of that contour
    x_min, y_min, w, h = cv2.boundingRect(largest_contour)
    x_max = x_min + w
    y_max = y_min + h
    
    return [x_min, y_min, x_max, y_max]

def get_roi_random_crop(image, bbox, crop_h, crop_w, roi_padding=10, random_position=True):
    h, w = image.shape[:2]
    x_min, y_min, x_max, y_max = bbox

    x_min = max(0, x_min - roi_padding)
    y_min = max(0, y_min - roi_padding)
    x_max = min(w, x_max + roi_padding)
    y_max = min(h, y_max + roi_padding)
    bbox_w, bbox_h = x_max - x_min, y_max - y_min

    if bbox_w > crop_w or bbox_h > crop_h:
        scale = min(crop_w / bbox_w, crop_h / bbox_h)
        new_w, new_h = max(1, round(w * scale)), max(1, round(h * scale))
        image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
        x_min, y_min, x_max, y_max = [v * scale for v in (x_min, y_min, x_max, y_max)]
        h, w = new_h, new_w
        bbox_w, bbox_h = x_max - x_min, y_max - y_min

    pad_b, pad_r = max(0, crop_h - h), max(0, crop_w - w)
    if pad_b or pad_r:
        image = cv2.copyMakeBorder(image, 0, pad_b, 0, pad_r, cv2.BORDER_CONSTANT, value=0)
        h, w = image.shape[:2]

    x_lo, x_hi = max(0, x_max - crop_w), min(x_min, w - crop_w)
    y_lo, y_hi = max(0, y_max - crop_h), min(y_min, h - crop_h)
    x_lo, x_hi = math.ceil(x_lo), math.floor(x_hi)
    y_lo, y_hi = math.ceil(y_lo), math.floor(y_hi)
    x_hi, y_hi = max(x_hi, x_lo), max(y_hi, y_lo)

    x_lo, x_hi = min(x_lo, w - crop_w), min(x_hi, w - crop_w)
    y_lo, y_hi = min(y_lo, h - crop_h), min(y_hi, h - crop_h)
    x_lo, y_lo = max(x_lo, 0), max(y_lo, 0)

    if random_position:
        x_start, y_start = random.randint(x_lo, x_hi), random.randint(y_lo, y_hi)
    else:
        x_start, y_start = (x_lo + x_hi) // 2, (y_lo + y_hi) // 2

    cropped = image[y_start:y_start + crop_h, x_start:x_start + crop_w]

    ch, cw = cropped.shape[:2]
    if ch != crop_h or cw != crop_w:
        cropped = cv2.copyMakeBorder(
            cropped, 0, max(0, crop_h - ch), 0, max(0, crop_w - cw),
            cv2.BORDER_CONSTANT, value=0
        )[:crop_h, :crop_w]

    return cropped



# -----------------------------------------
#              Dataset
# -----------------------------------------
try:
    import dicomsdl
except ImportError:
    dicomsdl = None
   

class MammoDatasetVariousAugPerView(Dataset):
    def __init__(self, df: pd.DataFrame, transforms: A.Compose = None, split: str = 'train',
                 max_images: int = MAX_IMAGES, random_crop_roi: bool = True):
        self.transforms = transforms
        self.split = split
        self.max_images = max_images
        self.random_crop_roi = random_crop_roi
        self.img_dir = (
            "/kaggle/input/competitions/rsna-breast-cancer-detection/train_images"
            if split == 'train' else
            "/kaggle/input/competitions/rsna-breast-cancer-detection/test_images"
        )
        self.records = df.to_dict(orient='records')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        patient_id = row['patient_id']
        image_ids = row['image_id']

        # ASSUMPTION: row['view'] is a list parallel to image_id, e.g. ['CC','MLO'].
        # If your df instead stores a single scalar 'view' per row, this breaks — confirm structure.
        views_raw = row.get('view', None)
        if views_raw is None:
            views_raw = ['CC'] * len(image_ids)
        elif not isinstance(views_raw, (list, tuple, np.ndarray)):
            views_raw = [views_raw]

        n_real = min(len(image_ids), self.max_images)
        images, view_ids = [], []

        for i in range(n_real):
            try:
                img_id = image_ids[i]
                img_path = os.path.join(self.img_dir, str(patient_id), f"{img_id}.dcm")
                try:
                    dcm = dicomsdl.open(img_path)
                    image = dcm.pixelData()
                    photometric = dcm.PhotometricInterpretation
                    try:
                        dcm_meta = pydicom.dcmread(img_path, stop_before_pixels=True)
                    except Exception:
                        dcm_meta = None
                except Exception:
                    try:
                        dcm = pydicom.dcmread(img_path)
                        image = dcm.pixel_array
                        photometric = getattr(dcm, 'PhotometricInterpretation', 'MONOCHROME2')
                        dcm_meta = dcm
                    except FileNotFoundError:
                        print(f'FileNotFoundError: {img_path}')
                        raise
    
                image = normalizer(image, photometric, dcm_meta=dcm_meta)
                bbox = row['bbox'][i]
                if bbox is None:
                    bbox = get_anatomy_bbox(image, threshold_value=15)
                tight_image = get_roi_random_crop(
                    image, bbox, crop_h=2*IMG_SIZE, crop_w=IMG_SIZE,
                    roi_padding=10, random_position=self.random_crop_roi
                )
            except Exception as e:
                print(f"[warn] skipping bad image {img_path}: {e}")
                image = np.zeros((2*IMG_SIZE, IMG_SIZE), dtype=np.uint8)
                tight_image = image
                
            if self.transforms:
                augmented = self.transforms(image=tight_image)
                image = augmented['image']
                if not isinstance(image, torch.Tensor):
                    image = torch.tensor(image, dtype=torch.float32) / 255.0
                    if image.ndim == 2:
                        image = image.unsqueeze(0)
            else:
                image = tight_image.astype(np.float32) / 255.0
                image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)

            images.append(image)                                   # (1, H, W)
            view_str = views_raw[i] if i < len(views_raw) else 'CC'
            view_ids.append(VIEW_TYPE_MAP.get(view_str, 0))

        # --- Pad: cycle-repeat real (image, view_id) pairs, no zeros, no re-decode ---
        valid_mask = [1] * n_real + [0] * (self.max_images - n_real)
        for i in range(n_real, self.max_images):
            if n_real > 0:
                src = i % n_real
                images.append(images[src].clone())
                view_ids.append(view_ids[src])
            else:
                # Rare: breast side with zero images at all
                images.append(torch.zeros((1, 2*IMG_SIZE, IMG_SIZE), dtype=torch.float32))
                view_ids.append(0)

        images_tensor = torch.stack(images, dim=0)   # (max_images, 1, H, W) <- stack, not cat

        data_packet = {
            'image': images_tensor,
            'view_type_ids': torch.tensor(view_ids, dtype=torch.long),   # (max_images,)
            'valid_mask': torch.tensor(valid_mask, dtype=torch.bool),    # (max_images,)
            'patient_id': patient_id,
            'laterality': row['laterality'],
        }

        # exclude_cols = {'patient_id', 'image_id', 'biopsy', 'invasive', 'density', 'laterality',
        #                 'BIRADS', 'difficult_negative_case', 'cancer', 'prediction_id', 'view'}
        # metadata = [float(row[k]) for k in row.keys() if k not in exclude_cols and pd.notna(row[k])]

        if self.split == 'train':
            data_packet['target'] = torch.tensor(row['cancer'], dtype=torch.float32)
        else:
            data_packet['prediction_id'] = str(row.get('prediction_id', f"{patient_id}_{row['laterality']}"))

        return data_packet

# -----------------------------------------
#                 Model 
# -----------------------------------------


class ImageEncoderPerView(nn.Module):
    def __init__(self, model_name='efficientnetv2_rw_m', pretrained=True):
        super().__init__()
        if model_name == 'efficientnetv2_rw_m':
            self.backbone = timm.create_model(
                model_name,
                pretrained=True,
                in_chans=1,
                num_classes=0,
                pretrained_cfg_overlay=dict(file='/kaggle/input/datasets/malihealavi/efficientnetv2-rw-m-weights/efficientnetv2_rw_m.pth')
            )
        self.num_features = self.backbone.num_features
       

    # def forward(self, images, metadata=None):
    #     stacked_encoding_vec = torch.cat([self.backbone(images[i, :, :]) for i in range(images.shape[0])], dim=0)
    def forward(self, images):
        # images: (B, MAX_VIEWS, 1, H, W)
        B, N, C, H, W = images.shape
        # feats = self.backbone(images.view(B * N, C, H, W))   # (B*N, D)
        x = images.view(B * N, C, H, W)
        feats = self.backbone(x)          
        return feats.view(B, N, self.num_features)

class ViewSetAggregator(nn.Module):
    """Variable-N set of per-view encodings -> single logit."""
    def __init__(self, feat_dim, n_view_types=6, attn_dim=128,
                 use_self_attn=True, n_heads=4, n_layers=1, dropout=0.3):
        super().__init__()
        self.view_type_embed = nn.Embedding(n_view_types, feat_dim)  # 0=CC, 1=MLO
        self.use_self_attn = use_self_attn
        if use_self_attn:
            layer = nn.TransformerEncoderLayer(
                d_model=feat_dim, nhead=n_heads, dim_feedforward=feat_dim * 2,
                dropout=dropout, batch_first=True, activation='gelu', norm_first=True,
            )
            self.self_attn = nn.TransformerEncoder(layer, num_layers=n_layers)

        # gated attention MIL pooling
        self.attn_V = nn.Linear(feat_dim, attn_dim)
        self.attn_U = nn.Linear(feat_dim, attn_dim)
        self.attn_w = nn.Linear(attn_dim, 1)

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, x, view_type_ids, valid_mask):
        # x: (B, MAX_VIEWS, D); view_type_ids: (B, MAX_VIEWS) long
        # valid_mask: (B, MAX_VIEWS) bool, True = real image
        x = x + self.view_type_embed(view_type_ids)

        if self.use_self_attn:
            x = self.self_attn(x, src_key_padding_mask=~valid_mask)  # True = ignore

        A_V = torch.tanh(self.attn_V(x))
        A_U = torch.sigmoid(self.attn_U(x))
        scores = self.attn_w(A_V * A_U).squeeze(-1)            # (B, MAX_VIEWS)
        scores = scores.masked_fill(~valid_mask, -1e4)         # -1e4, not -inf (NaN risk on TPU)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)

        pooled = (weights * x).sum(dim=1)
        logit = self.head(pooled).squeeze(-1)
        return logit, weights.squeeze(-1)

class ViewSetPooling(nn.Module):
    """Drop-in replacement for ViewSetAggregator — same forward signature,
    (B, N, D) x (B, N) view_type_ids x (B, N) valid_mask -> (logit, weights).
    Replaces the transformer + gated-attention MIL with simple pooling
    over the per-view feature vectors.
    """
    def __init__(self, feat_dim, n_view_types=6, dropout=0.3,
                 pool_type='gem', gem_p=3.0, learnable_p=False):
        super().__init__()
        self.pool_type = pool_type  # 'max' | 'mean' | 'gem'
        self.view_type_embed = nn.Embedding(n_view_types, feat_dim)
        if pool_type == 'gem':
            init_p = torch.tensor([gem_p], dtype=torch.float32)
            self.gem_p = nn.Parameter(init_p) if learnable_p else init_p

        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, x, view_type_ids, valid_mask):
        x = x + self.view_type_embed(view_type_ids)          # (B, N, D)
        mask = valid_mask.unsqueeze(-1)                        # (B, N, 1)
        counts = valid_mask.sum(dim=1, keepdim=True).clamp(min=1).float()

        if self.pool_type == 'max':
            pooled = x.masked_fill(~mask, float('-inf')).max(dim=1).values
            pooled = torch.nan_to_num(pooled, neginf=0.0)      # guards the rare all-invalid row

        elif self.pool_type == 'mean':
            pooled = x.masked_fill(~mask, 0.0).sum(dim=1) / counts

        elif self.pool_type == 'gem':
            p = self.gem_p.to(x.device).clamp(min=1.0, max=8.0)  # keep p in a stable range
            x_pow = x.clamp(min=1e-6).pow(p)                    # GeM needs non-negative input
            pooled = (x_pow.masked_fill(~mask, 0.0).sum(dim=1) / counts).clamp(min=1e-6).pow(1.0 / p)

        else:
            raise ValueError(f"unknown pool_type {self.pool_type}")

        logit = self.head(pooled).squeeze(-1)
        weights = None  
        return logit, weights
        
class MammoModelHybrid(nn.Module):
    def __init__(self, model_name='efficientnetv2_rw_m'):
        super().__init__()
        self.encoder = ImageEncoderPerView(model_name)
        # self.aggregator = ViewSetAggregator(feat_dim=self.encoder.num_features)
        self.aggregator = ViewSetPooling(feat_dim=self.encoder.num_features, pool_type='max') 
    def forward(self, images, view_type_ids, valid_mask):
        feats = self.encoder(images)
        return self.aggregator(feats, view_type_ids, valid_mask)


def freeze_bn(model):
    for m in model.encoder.backbone.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False
# -----------------------------------------
#                 Eval Metric 
# -----------------------------------------

def optimal_prob_f1_score(y_true, y_pred, beta=1):
    y_true = np.array(y_true).astype(np.float32)
    y_pred = np.array(y_pred).astype(np.float32)
    
    # Clip predictions to valid probability range, same as pfbeta
    y_pred = np.clip(y_pred, 0, 1)
    
    tp = np.sum(y_true * y_pred)
    fp = np.sum((1 - y_true) * y_pred)
    fn = np.sum(y_true * (1 - y_pred))
    
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    
    f1 = (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall + 1e-8)
    return f1

def f2_best_threshold(all_targets, all_probs):
    """Finds the threshold that maximizes the F2 score (favors recall)."""
    precisions, recalls, thresholds = precision_recall_curve(all_targets, all_probs)
    p = precisions[:-1]
    r = recalls[:-1]
    
    # Safely calculate F2 scores for all thresholds
    with np.errstate(divide='ignore', invalid='ignore'):
        f2_scores = (5 * p * r) / ((4 * p) + r)
        f2_scores = np.nan_to_num(f2_scores)

    best_f2_idx = np.argmax(f2_scores)
    best_threshold_f2 = thresholds[best_f2_idx] # Fixed NameError
    
    return best_threshold_f2, f2_scores[best_f2_idx]


def f1_best_threshold(all_targets, all_probs):
    """Finds the threshold that maximizes the F1 score (balances precision & recall)."""
    precisions, recalls, thresholds = precision_recall_curve(all_targets, all_probs)
    p = precisions[:-1]
    r = recalls[:-1]
    
    # Safely calculate F1 scores for all thresholds
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores = (2 * p * r) / (p + r)
        f1_scores = np.nan_to_num(f1_scores)

    best_f1_idx = np.argmax(f1_scores)
    best_threshold_f1 = thresholds[best_f1_idx]
    
    return best_threshold_f1, f1_scores[best_f1_idx]
# -----------------------------------------
#               plot loss 
# -----------------------------------------

def plot_loss(val_losses, train_losses, val_f1 , val_f2, path):
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, label='Training Loss', color='blue', marker='o', linewidth=2)
    plt.plot(epochs, val_f1, label='Valid F1', color='green', marker='o', linewidth=2)
    plt.plot(epochs, val_losses, label='Validation Loss', color='red', marker='s', linewidth=2)
    plt.plot(epochs, val_f2, label='Valid F2', color='yellow', marker='s', linewidth=2)
    
    plt.title('Training and Validation Loss Over Epochs', fontsize=14)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss/Eval', fontsize=12)
    plt.xticks(epochs)
    plt.legend(fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plot_path = path
    plt.savefig(plot_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    
    print(f"Plot successfully saved to {os.path.abspath(plot_path)}")


# -----------------------------------------
#                 Helper
# -----------------------------------------
def unwrap(m):
    """DataParallel stores the real model under `.module` — use this
    whenever you need to reach a custom attribute (e.g. .backbone)."""
    return m.module if isinstance(m, nn.DataParallel) else m
    
def to_scalar(x):
    """Works for both tensor and plain number."""
    return x.item() if torch.is_tensor(x) else float(x)

def _to_cpu(obj):
    """Recursively move any torch.Tensor in a nested dict to CPU.
    Deliberately does NOT recurse into list/tuple/bytes."""
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu(v) for k, v in obj.items()}
    return obj

def move_batch(batch, device):
    """NEW: replaces ParallelLoader (which moved batches to the TPU for us)."""
    return {k: (v.to(device, non_blocking=True) if torch.is_tensor(v) else v)
            for k, v in batch.items()}

In [ ]:



USE_GRAD_CHECKPOINTING = True   # set False to turn it off
# NOTE: in the model, ImageEncoderPerView.forward must call  self.backbone(x)  directly
#       (remove the manual checkpoint(...) call, timm does the checkpointing for you)

USE_AMP = True                 # True = mixed precision (faster, less memory)
AMP_DTYPE = torch.float16       # T4 / P100 / V100: float16 (uses GradScaler)
                                # A100 / H100 / L4 / RTX 30xx+: torch.bfloat16 (no scaler needed)

NUM_WORKERS = 4   

# -----------------------------------------
#                 Trainer
# -----------------------------------------


class GPUMammoTrainer():
    def __init__(self, train_slice, val_slice, epochs, train_transforms, valid_transforms):
        # 1. Use the GPU
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.epochs = epochs
        self.accumulation_steps = ACCUMULATION_STEP
        self.train_slice = train_slice
        self.val_slice = val_slice
        # 2. Instantiate Model and move to GPU
        self.model = MammoModelHybrid(model_name='efficientnetv2_rw_m')
        self.model.to(self.device)
         
        if USE_GRAD_CHECKPOINTING:
            self.model.encoder.backbone.set_grad_checkpointing(True)

        # if torch.cuda.device_count() > 1:
        #     print(f"Using {torch.cuda.device_count()} GPUs")
        #     self.model = nn.DataParallel(self.model)
       
        # NEW: GradScaler is only needed for float16 (it does nothing when disabled)
        self.scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and AMP_DTYPE == torch.float16))

        #------------ Optimizer
        base = unwrap(self.model)
        backbone_params = base.encoder.backbone.parameters()
        head_params = base.aggregator.parameters()

        self.optimizer = optim.AdamW([
            {'params': backbone_params, 'lr': 1e-5},   # small, protect pretrained features
            {'params': head_params,     'lr': 1e-4},   # needs to learn from scratch
        ], weight_decay=1e-4)

        #------------- Scheduler

        warmup = LinearLR(self.optimizer, start_factor=0.05, total_iters=WARMUP_EPOCHS)
        cosine = CosineAnnealingLR(self.optimizer, T_max=self.epochs - WARMUP_EPOCHS, eta_min=1e-8)
        self.scheduler = SequentialLR(self.optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])

        #------------- Loss

        self.best_f1 = 0.0
        self.best_val_loss = float('inf')
        self.train_losses = []
        self.val_losses = []
        self.val_f1_scores = []
        self.val_f2_scores = []
        self.patience = 0
        self.start_epoch = 1
        self.neg_sampler_seed = 42

        #------------- predifine variables
        self.p_real = None
        self.p_train= None
        self.pos_weights= None
        self.criterion= None
        self.pr_resize = False
        self._load_checkpoint_if_exists()

    def logit_adjustment(self, raw_logits):
        return logit_adjustment(raw_logits, self.p_train, self.p_real, self.pos_weights)

    def _save_checkpoint(self, epoch, path):
        # single process: no master check, no rendezvous needed
        torch.save(_to_cpu(self._checkpoint_dict(epoch)), path)

    def _checkpoint_dict(self, epoch):
        return {
            'epoch': epoch,
            # 'model': self.model.state_dict(),            
            'model': unwrap(self.model).state_dict(),
            'optimizer': self.optimizer.state_dict(),
            'scheduler': self.scheduler.state_dict(),
            'best_f1': self.best_f1,
            'patience': self.patience,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'val_f1_scores': self.val_f1_scores,
            'val_f2_scores': self.val_f2_scores,
            'neg_sampler_state': self.neg_sampler.state_dict(),
        }

    def _load_checkpoint_if_exists(self, path="/kaggle/input/datasets/malihealavi/breast-cancer-gpu/last_checkpoint.pth"):
        if not os.path.exists(path):
            print(f"No checkpoint found at {path}, starting fresh.")
            return
        checkpoint = torch.load(path, map_location='cpu', weights_only=False)
        unwrap(self.model).load_state_dict(checkpoint['model'])
        self.optimizer.load_state_dict(checkpoint['optimizer'])

        if 'scheduler' in checkpoint:
            self.scheduler.load_state_dict(checkpoint['scheduler'])
            print('sceduler added')
        else:
            print("[warn] checkpoint has no scheduler state; scheduler starts fresh.")

        self.best_f1 = checkpoint.get('best_f1', 0.0)
        self.patience = checkpoint.get('patience', 0)
        self.start_epoch = checkpoint['epoch'] + 1

        self.train_losses = checkpoint.get('train_losses', [])
        self.val_losses = checkpoint.get('val_losses', [])
        self.val_f1_scores = checkpoint.get('val_f1_scores', [])
        self.val_f2_scores = checkpoint.get('val_f2_scores', [])
        self.neg_sampler_state = checkpoint.get('neg_sampler_state')
        print(
            f"Resumed from {path}: last completed epoch={checkpoint['epoch']}, "
            f"best_f1={self.best_f1:.4f}, patience={self.patience}, "
            f"resuming at epoch {self.start_epoch}"
        )
        global IMG_SIZE
        if IMG_SIZE == 1024 and self.pr_resize == True:   # new, bigger resolution run
            base = unwrap(self.model)
        
            # 1. Reset the "best" values (old size scores are not comparable)
            self.best_f1 = 0.0
            self.best_val_loss = float('inf')
            self.patience = 0
            self.val_losses = []
            self.val_f1_scores = []
            self.val_f2_scores = []
        
            # 2. New optimizer with lower LR (take params from the model again)
            self.optimizer = optim.AdamW([
                {'params': base.encoder.backbone.parameters(), 'lr': 5e-6},
                {'params': base.aggregator.parameters(),       'lr': 5e-5},
            ], weight_decay=1e-4)
        
            # 3. New scheduler for the NEW optimizer (short warmup + cosine)
            remaining = self.epochs - (self.start_epoch - 1)
            warmup = LinearLR(self.optimizer, start_factor=0.1, total_iters=1)
            cosine = CosineAnnealingLR(self.optimizer, T_max=max(remaining - 1, 1), eta_min=1e-8)
            self.scheduler = SequentialLR(self.optimizer, schedulers=[warmup, cosine], milestones=[1])
        
            # 4. Fresh grad scaler (only matters for float16)
            self.scaler = torch.amp.GradScaler('cuda', enabled=(USE_AMP and AMP_DTYPE == torch.float16))
            print(
            f"Resumed from {path}: last completed epoch={checkpoint['epoch']},\n "
            f"best_f1={self.best_f1:.4f}, patience={self.patience},\n "
            f"resuming at epoch {self.start_epoch}\n"
            f"img size {IMG_SIZE}\n"
        )

    def train_one_epoch(self, epoch):
        self.model.train()

        running_loss = 0.0

        iterator = tqdm(self.train_loader, desc=f"Epoch {epoch} [Train]")
        for i, batch in enumerate(iterator):
            batch = move_batch(batch, self.device)
            img = batch['image']
            mask = batch['valid_mask']
            view_type_id = batch['view_type_ids']
            targets = batch['target']

            self.optimizer.zero_grad()
            outputs, _ = self.model(img, view_type_id, mask)
            loss = self.criterion(outputs, targets)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

            self.optimizer.step()      # was: xm.optimizer_step(self.optimizer, barrier=True)

            running_loss += loss.detach() * img.size(0)

        return to_scalar(running_loss) / (len(self.train_loader.dataset) + 1e-8)


    def train_one_epoch_accumulated(self, epoch):
        self.model.train()
        freeze_bn(unwrap(self.model))
        running_loss = 0.0
        self.optimizer.zero_grad()
        iterator = tqdm(self.train_loader, desc=f"Epoch {epoch} [Train]")

        for i, batch in enumerate(iterator):
            batch = move_batch(batch, self.device)
            img, mask, view_type_id = batch['image'], batch['valid_mask'], batch['view_type_ids']
            targets = batch['target']

            with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                outputs, _ = self.model(img, view_type_id, mask)
            outputs = outputs.float()      # loss in float32
            loss = self.criterion(outputs, targets) / self.accumulation_steps
            self.scaler.scale(loss).backward()
            running_loss += loss.detach() * img.size(0) * self.accumulation_steps

            if (i + 1) % self.accumulation_steps == 0:
                self.scaler.unscale_(self.optimizer)     # needed so clipping sees the real gradients
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.scaler.step(self.optimizer)         # was: xm.optimizer_step(...)
                self.scaler.update()
                self.optimizer.zero_grad()

        # flush any leftover partial accumulation at end of epoch
        if (i + 1) % self.accumulation_steps != 0:
            self.scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad()

        return to_scalar(running_loss) / (len(self.train_loader.dataset) + 1e-8)


    def valid_one_epoch(self, epoch):
        self.model.eval()

        running_loss = torch.tensor(0.0, device=self.device)
        tp = torch.tensor(0.0, device=self.device)
        fp = torch.tensor(0.0, device=self.device)
        fn = torch.tensor(0.0, device=self.device)

        raw_probs_chunks = []
        targets_chunks = []

        iterator = tqdm(self.valid_loader, desc=f"Epoch {epoch} [Valid]")

        with torch.no_grad():
            for batch in iterator:
                batch = move_batch(batch, self.device)
                img = batch['image']
                mask = batch['valid_mask']
                view_type_id = batch['view_type_ids']
                targets = batch['target']
                with torch.autocast('cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
                    outputs, _ = self.model(img, view_type_id, mask)
                outputs = outputs.float()
                loss = self.criterion(outputs, targets)
                running_loss += loss.detach() * img.size(0)

                raw_probs = torch.sigmoid(outputs).float()
                # metrics reported here must match what submission will actually score,
                # so pF1/pF2 use corrected probs
                corrected_probs = torch.sigmoid(self.logit_adjustment(outputs)).float()

                tp += (targets * corrected_probs).sum()
                fp += ((1 - targets) * corrected_probs).sum()
                fn += (targets * (1 - corrected_probs)).sum()

                # AUC: rank-invariant to the shift, keep raw, gather once at epoch end
                raw_probs_chunks.append(raw_probs.detach())
                targets_chunks.append(targets.detach())

        # single GPU: no mesh_reduce needed, these are already the full totals
        running_loss = to_scalar(running_loss)
        tp = to_scalar(tp)
        fp = to_scalar(fp)
        fn = to_scalar(fn)

        epoch_loss = running_loss / (len(self.valid_loader.dataset) + 1e-8)
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        epoch_f1 = 2 * precision * recall / (precision + recall + 1e-8)
        epoch_f2 = 5 * tp / (5 * tp + 4 * fn + fp + 1e-8)   # pFbeta, beta=2, same tp/fp/fn

        # single GPU: no all_gather needed, just join the chunks
        global_probs = torch.cat(raw_probs_chunks).cpu().numpy().ravel()
        global_targets = torch.cat(targets_chunks).cpu().numpy().ravel()

        f2_best_trsh, best_f2_score = f2_best_threshold(global_targets, global_probs)
        f1_best_trsh, best_f1_score = f1_best_threshold(global_targets, global_probs)

        print(f"Global Optimal F1 Threshold: {f1_best_trsh:.4f} (Score: {best_f1_score:.4f})")
        print(f"Global Optimal F2 Threshold: {f2_best_trsh:.4f} (Score: {best_f2_score:.4f})")

        return epoch_loss, epoch_f1, epoch_f2


    def training(self):
        print(f"Starting Training on GPU...")
        patient_targets_full = self.train_slice.groupby('patient_id')['cancer'].max()
        neg_ids_full = patient_targets_full[patient_targets_full == 0].index.tolist()
        self.neg_sampler = BalancedPatientSampler(neg_ids_full, seed=self.neg_sampler_seed)
        if getattr(self, 'neg_sampler_state', None) is not None:
            self.neg_sampler.load_state_dict(self.neg_sampler_state)

        for epoch in range(self.start_epoch, self.epochs + 1):

            # -------------- random DownSampling every epoch
            self.p_real = self.train_slice['cancer'].mean()
            train_sub = create_balanced_patient_subset(self.train_slice, self.neg_sampler, NEG_TO_POS_RATIO)
            self.p_train = train_sub['cancer'].mean()
            self.pos_weights = (train_sub['cancer'] == 0).sum() / (train_sub['cancer'] == 1).sum()
            pos_weight_tensor = torch.tensor([self.pos_weights], dtype=torch.float32).to(self.device)

            self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
            train_ds = MammoDatasetVariousAugPerView(train_sub, transforms=train_transforms, split='train', random_crop_roi=True)
            val_ds   = MammoDatasetVariousAugPerView(self.val_slice, transforms=valid_transforms, split='train', random_crop_roi=False)
            print(f"\n Epoch {epoch} : p_real:{self.p_real:.4f} | p_train:{self.p_train:.4f} | pos_weights:{self.pos_weights:.4f} | length train_sub: {len(train_sub)} | length val: {len(self.val_slice)} | train_sub target count: {train_sub['cancer'].value_counts()} | batch_size = {BATCH_SIZE}")

            # --------------- Data Loaders (no DistributedSampler on one GPU)
            # generator seeded with epoch = same idea as sampler.set_epoch(epoch)
            g = torch.Generator()
            g.manual_seed(42 + epoch)

            self.train_loader = DataLoader(
                dataset=train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                shuffle=True, generator=g, drop_last=True, pin_memory=True,
            )
            self.valid_loader = DataLoader(
                dataset=val_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                shuffle=False, drop_last=False, pin_memory=True,
            )
            # ----------------------------------------------------------
            tr_loss = self.train_one_epoch_accumulated(epoch)
            val_loss, val_f1_score, val_f2 = self.valid_one_epoch(epoch)

            # --- record history ---
            self.train_losses.append(tr_loss)
            self.val_losses.append(val_loss)
            self.val_f1_scores.append(val_f1_score)
            self.val_f2_scores.append(val_f2)

            print(f"\nEpoch {epoch} Summary: Train Loss = {tr_loss:.4f} | Valid Loss = {val_loss:.4f} | Valid Prob F1 = {val_f1_score:.4f} | Valid F2 = {val_f2:.4f} | LR = {self.optimizer.param_groups[0]['lr']:.6f}\n")

            self.scheduler.step()

            if epoch <= WARMUP_EPOCHS:
                # model isn't at full LR yet — don't let a noisy/weak warmup epoch
                # burn patience budget before real training has started
                if val_f1_score > self.best_f1:
                    self.best_f1 = val_f1_score
                    self._save_checkpoint(epoch, "best_model_checkpoint.pth")
                self.patience = 0
            else:
                if val_f1_score > self.best_f1:
                    self.best_f1 = val_f1_score
                    self._save_checkpoint(epoch, "best_model_checkpoint.pth")
                    print(f"Valid F1 improved to {val_f1_score:.4f}. Saved.\n")
                    self.patience = 0
                else:
                    self.patience += 1
            self._save_checkpoint(epoch, "last_checkpoint.pth")

            if self.patience >= PATIENCE:
                print("Early stopping triggered.")
                break

            del self.train_loader
            del self.valid_loader
            gc.collect()
            torch.cuda.empty_cache()     # NEW: free GPU memory between epochs
            time.sleep(2)

        plot_loss(
            self.val_losses,
            self.train_losses,
            self.val_f1_scores,
            self.val_f2_scores,
            "/kaggle/working/loss_f1_plot.png"
        )

# -----------------------------------------
#                 Execution
# -----------------------------------------

def run(train_slice, val_slice, train_transforms, valid_transforms):
    torch.manual_seed(42)       # was: xm.set_rng_state(42)
    try:
        trainer = GPUMammoTrainer(
            train_slice=train_slice, val_slice=val_slice, epochs=50,
            train_transforms=train_transforms, valid_transforms=valid_transforms
        )
        trainer.training()
    except Exception:
        import traceback
        with open("/kaggle/working/crash.log", "w") as f:
            traceback.print_exc(file=f)
        raise

if __name__ == '__main__':

    train_df = pd.read_csv('/kaggle/input/datasets/malihealavi/breast-cancer-gpu/train_with_bbox.csv')
    bbox_cols = ['bbox_x_min', 'bbox_y_min', 'bbox_x_max', 'bbox_y_max']
    train_df['bbox'] = train_df[bbox_cols].apply(
        lambda row: row.tolist() if row.notna().all() else None,
        axis=1
    )
    train_df.to_csv('/kaggle/working/train_with_bbox.csv', index=False)
    train_df['patient_id'] = train_df['patient_id'].astype(str)
    train_df['image_id'] = train_df['image_id'].astype(str)
    print(f"length of original df : {len(train_df)}")
    train_df = group_views_by_patient_laterality(train_df)
    print(f"length of df after grouping on patient_id and laterality : {len(train_df)}")

    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=42)
    folds = sgkf.split(X=train_df, y=train_df['cancer'], groups=train_df['patient_id'])
    train_idx, val_idx = next(folds)
    train_slice = train_df.iloc[train_idx].reset_index(drop=True)
    val_slice = train_df.iloc[val_idx].reset_index(drop=True)
    print(f"length train_set: {len(train_slice)} , length val_set: {len(val_slice)}")
    print(f"train value_count(cancer) : {train_slice['cancer'].value_counts()}")
    print(f" val value_count(cancer): {val_slice['cancer'].value_counts()}")

    run(train_slice, val_slice, train_transforms, valid_transforms)  

# Save bbox ROI

In [ ]:
!pip install --no-index --find-links=/kaggle/input/datasets/malihealavi/pydicom-packages-rsna-breast-cancer pydicom pylibjpeg pylibjpeg-libjpeg

import warnings
warnings.filterwarnings("ignore")
import os
import sys
import json
from glob import glob
import random
import collections
import time
import re
from typing import Optional
import math
import pickle
from sklearn.metrics import precision_recall_curve

import numpy as np
import pandas as pd
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

import torch
from torch import nn
from torch.utils import data as torch_data
from sklearn import model_selection as sk_model_selection
from torch.nn import functional as torch_functional
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data.distributed import DistributedSampler

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.model_selection import train_test_split
from IPython.display import clear_output
clear_output()

!pip install dicomsdl
clear_output()

import dicomsdl  # fixes silent pydicom-only fallback

def normalizer(image, photometric, dcm_meta=None):
    image = image.astype(np.float32)

    if dcm_meta is not None:
        try:
            image = apply_voi_lut(image, dcm_meta)
        except Exception:
            pass  # missing/malformed VOI LUT tags -> fall back to raw values

    if photometric == 'MONOCHROME1':
        image = image.max() - image   # invert AFTER windowing

    image_min, image_max = np.min(image), np.max(image)
    if image_max - image_min > 0:
        image = (image - image_min) / (image_max - image_min) * 255.0
    else:
        image = np.zeros_like(image)
    return image.astype(np.uint8)
    
def get_anatomy_bbox(image, threshold_value=15):
    """
    Finds the bounding box of the main anatomy in a normalized 8-bit image
    by isolating the largest connected component.
    
    Returns:
        bbox (list): [x_min, y_min, x_max, y_max]
    """
    # 1. Light Gaussian blur to smooth out internal textures and noise
    blurred = cv2.GaussianBlur(image, (5, 5), 0)
    
    # 2. Binarize the image
    # Any pixel brighter than threshold_value becomes 255 (white), rest become 0 (black)
    _, binary_mask = cv2.threshold(blurred, threshold_value, 255, cv2.THRESH_BINARY)
    
    # 3. Clean up the mask using Morphological Closing
    # This fills in small holes inside the anatomy (like dark spots in lungs or ventricles)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    closed_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
    
    # 4. Find all external contours in the mask
    contours, _ = cv2.findContours(closed_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Fallback if image is completely black or no contours found
    if not contours:
        h, w = image.shape
        return [0, 0, w, h]
        
    # 5. Extract the largest contour by area (this drops small text/markers)
    largest_contour = max(contours, key=cv2.contourArea)
    
    # 6. Get the standard bounding box of that contour
    x_min, y_min, w, h = cv2.boundingRect(largest_contour)
    x_max = x_min + w
    y_max = y_min + h
    
    return [x_min, y_min, x_max, y_max]
    
class DataROI(Dataset):
    def __init__(self, df: pd.DataFrame, random_crop_roi: bool = True):
        self.random_crop_roi = random_crop_roi
        self.img_dir = (
            "/kaggle/input/competitions/rsna-breast-cancer-detection/train_images"
        )
        self.records = df.to_dict(orient='records')

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]
        patient_id = row['patient_id']
        image_ids = row['image_id']

        # ASSUMPTION: row['view'] is a list parallel to image_id, e.g. ['CC','MLO'].
        # If your df instead stores a single scalar 'view' per row, this breaks — confirm structure.
        
        images, view_ids = [], []
        try:
            img_id = image_ids
            img_path = os.path.join(self.img_dir, str(patient_id), f"{img_id}.dcm")
            try:
                dcm = dicomsdl.open(img_path)
                image = dcm.pixelData()
                photometric = dcm.PhotometricInterpretation
                try:
                    dcm_meta = pydicom.dcmread(img_path, stop_before_pixels=True)
                except Exception:
                    dcm_meta = None
            except Exception:
                try:
                    dcm = pydicom.dcmread(img_path)
                    image = dcm.pixel_array
                    photometric = getattr(dcm, 'PhotometricInterpretation', 'MONOCHROME2')
                    dcm_meta = dcm
                except FileNotFoundError:
                    print(f'FileNotFoundError: {img_path}')
                    raise

            image = normalizer(image, photometric, dcm_meta=dcm_meta)
            bbox = get_anatomy_bbox(image, threshold_value=15)
            
            return bbox , patient_id , image_ids
        except Exception as e:
            print(f"[warn] skipping bad image {img_path}: {e}")

train_df = pd.read_csv('/kaggle/input/competitions/rsna-breast-cancer-detection/train.csv')
train_ds = DataROI(train_df, random_crop_roi=True)

cv2.setNumThreads(0)
os.environ['OMP_NUM_THREADS'] = '1'

def collate_skip_none(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return torch_data.dataloader.default_collate(batch)

train_loader = DataLoader(
    dataset=train_ds, batch_size=32, num_workers=2,
    drop_last=False, timeout=300, collate_fn=collate_skip_none,
)

bbox_map = {}  # (patient_id, image_id) -> [x_min, y_min, x_max, y_max]

for batch in tqdm(train_loader, total=len(train_loader)):
    if batch is None:
        continue
    bb, patient_id, image_ids = batch
    # bbox is returned as a 4-elem list per sample -> default_collate transposes
    # it into a list of 4 tensors, each shape (B,), NOT a list of per-sample bboxes.
    x_min, y_min, x_max, y_max = bb

    patient_id = patient_id.tolist() if torch.is_tensor(patient_id) else list(patient_id)
    image_ids = image_ids.tolist() if torch.is_tensor(image_ids) else list(image_ids)

    for i in range(len(patient_id)):
        pid, iid = patient_id[i], image_ids[i]
        bbox_map[(pid, iid)] = [int(x_min[i]), int(y_min[i]), int(x_max[i]), int(y_max[i])]

# key on (patient_id, image_id) rather than position -> robust to dropped/None samples
train_df['bbox'] = train_df.apply(
    lambda r: bbox_map.get((r['patient_id'], r['image_id'])), axis=1
)

n_missing = train_df['bbox'].isna().sum()
print(f'{n_missing} / {len(train_df)} rows missing bbox (failed decode)')

# pickle preserves the list dtype; CSV is flattened for readability
train_df.to_pickle('/kaggle/working/train_with_bbox.pkl')

bbox_cols = pd.DataFrame(
    train_df['bbox'].apply(lambda b: b if isinstance(b, list) else [None]*4).tolist(),
    columns=['bbox_x_min', 'bbox_y_min', 'bbox_x_max', 'bbox_y_max'],
    index=train_df.index,
)
train_df.drop(columns='bbox').join(bbox_cols).to_csv('/kaggle/working/train_with_bbox.csv', index=False)
